# Fraud Detection — Training Pipeline

Full pipeline from raw transactions to a trained Isolation Forest.

## Steps
1. **Load** clean synthetic transactions (22.7M rows, 10k users)
2. **Inject fraud** — 7 scenario types via `FraudInjector` → 2,500 scenarios, ~15k rows
3. **Compute features** — 13 transaction-level features via `compute_features_batch` (DuckDB + pandas)
4. **Inspect** — per-scenario fingerprint table; verify feature distributions
5. **Train** — Isolation Forest (unsupervised; labels not used in fit)
6. **Evaluate** — precision@1000, per-scenario recall, score distribution
7. **Save** model to `models/fraud_model.pkl`

> **Architecture note**: The model is unsupervised. `is_fraud` labels are injected
> *after* training and used only for evaluation — we never leak them into `model.fit()`.

In [ ]:
import sys, time, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 60)

DATA_DIR   = Path('../data')
MODELS_DIR = Path('../models')
print('Ready.')

## Step 1 — Load Clean Transactions

22.7M inflow transactions for 10,000 synthetic Nigerian trader profiles.
Generated by `training/synthetic_data_gen.py` with 6 archetypes (market vendor,
electronics retailer, tailor, okada rider, POS agent, beauty supplier).

In [ ]:
t0 = time.time()
clean_df = pd.read_parquet(DATA_DIR / 'synth_transactions.parquet')
clean_df['occurred_at'] = pd.to_datetime(clean_df['occurred_at'])

print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Rows:  {len(clean_df):,}')
print(f'Users: {clean_df.user_id.nunique():,}')
print(f'Date range: {clean_df.occurred_at.min().date()} → {clean_df.occurred_at.max().date()}')
print()
print('Schema:')
print(clean_df.dtypes.to_string())
print()
clean_df.head(3)

## Step 2 — Inject Fraud Scenarios

`FraudInjector` generates 7 synthetic fraud scenario types:

| Scenario | Weight | Description |
|---|---|---|
| `score_pump` | 35% | Burst of novel senders to inflate credit features |
| `large_novel_deposit` | 20% | Single 10-25× deposit from new sender |
| `velocity_known_sender` | 15% | 10-20 txns from one known sender in 1 hour |
| `round_trip` | 10% | Inflow immediately reversed by outflow |
| `amount_cloning` | 10% | Same exact amount from 4-8 different senders |
| `dormancy_spike` | 5% | Silent user suddenly floods txns |
| `round_number_flood` | 5% | Flood of suspiciously round kobo amounts |

The injector reads user baselines from clean data so fraud amounts/timings are
calibrated to each user's real history — not random noise.

In [ ]:
from training.fraud_injector import FraudInjector, InjectionConfig

fraud_path = DATA_DIR / 'synth_transactions_fraud.parquet'

if fraud_path.exists():
    print('Loading pre-generated fraud scenarios...')
    fraud_df = pd.read_parquet(fraud_path)
else:
    print('Generating fraud scenarios (takes ~10 min on 22.7M rows)...')
    t0 = time.time()
    injector = FraudInjector(clean_df, InjectionConfig(target_scenarios=2500))
    fraud_df  = injector.generate_all()
    fraud_df.to_parquet(fraud_path, index=False)
    print(f'Done in {time.time()-t0:.0f}s')

fraud_df['occurred_at'] = pd.to_datetime(fraud_df['occurred_at'])
print(f'Fraud rows:      {len(fraud_df):,}')
print(f'Fraud scenarios: {fraud_df.fraud_scenario_id.nunique():,}')
print()
print('By scenario type:')
print(fraud_df.groupby('fraud_type')['fraud_scenario_id'].nunique().sort_values(ascending=False).to_string())

## Step 3 — Compute 13 Fraud Features

`compute_features_batch` computes per-transaction features using only
**prior** transactions for the same user (point-in-time correct).

**Feature families:**
- **Amount** (3): z-score vs user baseline, log-ratio to median, round-number flag
- **Velocity** (2): txn count in last 1h / 6h
- **Timing** (3): time since last txn, hour rarity, day-of-week rarity
- **Sender** (3): novelty flag, days since first seen, 24h sender concentration
- **Cross-sender** (2): exact-amount match in 24h, inflow/outflow reciprocity

**Pipeline**: expanding stats (pandas) → first-seen (groupby-transform)
→ time-windowed counts (DuckDB interval joins) → self-joins (DuckDB)

In [ ]:
from training.fraud_feature_engine import compute_features_batch, FraudFeatures

features_path = DATA_DIR / 'features_train.parquet'

if features_path.exists():
    print('Loading pre-computed features...')
    t0       = time.time()
    feat_df  = pd.read_parquet(features_path)
    print(f'Loaded {len(feat_df):,} rows in {time.time()-t0:.1f}s')
else:
    print('Computing features on clean + fraud union...')
    # Union: keep only the columns that appear in both DataFrames
    shared_cols = ['user_id', 'occurred_at', 'amount_kobo', 'sender_name', 'type',
                   'is_fraud', 'fraud_type', 'fraud_scenario_id']

    clean_for_union        = clean_df.copy()
    clean_for_union['is_fraud']           = False
    clean_for_union['fraud_type']         = None
    clean_for_union['fraud_scenario_id']  = None

    combined = pd.concat(
        [clean_for_union[shared_cols],
         fraud_df[shared_cols]],
        ignore_index=True,
    )

    print(f'Union: {len(combined):,} rows  '
          f'(fraud rate: {combined.is_fraud.mean():.4%})')

    t0      = time.time()
    feat_df = compute_features_batch(combined)
    elapsed = time.time() - t0
    print(f'Done in {elapsed:.0f}s  ({len(feat_df):,} rows)')

    feat_df.to_parquet(features_path, index=False)
    print(f'Saved → {features_path}')

print()
print(f'Features: {FraudFeatures.FEATURE_NAMES}')

## Step 4 — Inspect Feature Distributions

Two checks before training:

1. **Distribution sanity** — no unexpected NaNs, no extreme unclipped values
2. **Scenario fingerprints** — each fraud type should have a distinct signal pattern.
   If two scenarios have identical means across all 13 features, the model can't separate them.

In [ ]:
print('=== Feature distributions ===')
print(feat_df[FraudFeatures.FEATURE_NAMES].describe().round(3).to_string())
print()
print('NaN counts:')
print(feat_df[FraudFeatures.FEATURE_NAMES].isna().sum().to_string())

In [ ]:
print('=== 13-feature fingerprint by scenario ===')
print('Each scenario should have a unique pattern. Two identical rows = feature design failure.')
print()
print(feat_df.groupby('fraud_type')[FraudFeatures.FEATURE_NAMES].mean().round(2).to_string())

In [ ]:
# Visualise score distributions for 3 key features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

clean_rows = feat_df[~feat_df['is_fraud'].fillna(False)]
fraud_rows = feat_df[feat_df['is_fraud'].fillna(False)]

for ax, feat in zip(axes, ['amount_zscore_user', 'txn_count_1h', 'sender_novelty']):
    ax.hist(clean_rows[feat].clip(-10, 10), bins=50, alpha=0.6, label='clean', density=True)
    ax.hist(fraud_rows[feat].clip(-10, 10), bins=50, alpha=0.6, label='fraud', density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Clean vs Fraud Feature Distributions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/fraud_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/fraud_feature_distributions.png')

## Step 5 — Train Isolation Forest

**Why Isolation Forest?**
- Unsupervised: works without labelled data (critical in production where fraud labels are sparse)
- Linear time complexity: O(n × n_estimators × log(max_samples))
- Naturally handles the anomaly detection framing: fraud is rare and structurally different

**Key parameters:**
- `contamination=0.005` — tells the model ~0.5% of data is suspicious. Set to ~7× actual fraud rate (0.07%) to give headroom for genuinely-weird clean transactions.
- `max_samples=256` — each tree sub-samples 256 rows. Default sklearn value; enough to capture normal-data structure at 22M scale.
- `n_estimators=200` — ensemble size. More = more stable; 200 gives <20ms single-row predict.

**The model does NOT see `is_fraud` labels during fit.**

In [ ]:
from training.fraud_model import train, TrainConfig, FEATURE_NAMES

config = TrainConfig(
    contamination=0.005,
    n_estimators=200,
    max_samples=256,
    random_state=42,
)

print(f'Training on {len(feat_df):,} rows × {len(FEATURE_NAMES)} features...')
print(f'Config: {config}')
print()

t0    = time.time()
model = train(feat_df, config)
elapsed = time.time() - t0

print(f'Fit time: {elapsed:.1f}s')
print(f'Model: {model}')

## Step 6 — Evaluate

**Evaluation methodology for unsupervised fraud detection:**

We can't use standard accuracy/F1 because the model was trained without labels.
Instead we use:

1. **Score distribution** — are clean and fraud rows separating in score space?
2. **Precision@1000** — of the 1,000 highest-scored transactions, what fraction are fraud?
   A random baseline would give ~0.07% (the actual fraud rate). Good model: >50%.
3. **Per-scenario recall@1000** — which fraud scenarios does the model catch?
   Reveals which features are working and which need iteration.
4. **Top-1000 is_fraud rate** — final sanity check: are the top anomalies actually fraud?

In [ ]:
from training.fraud_model import evaluate, score_samples

eval_results = evaluate(model, feat_df, top_k=1000)

print('=== Score distribution ===')
for k, v in eval_results['score_distribution'].items():
    print(f'  {k:<8} {v:.4f}')

print()
print(f'Precision@1000:  {eval_results["precision_at_1000"]:.1%}')
print(f'AUC-PR (approx): {eval_results["auc_pr_approx"]:.3f}')
print()
print('Recall@1000 by scenario:')
for scenario, recall in sorted(eval_results['recall_by_scenario'].items(), key=lambda x: -x[1]):
    bar = '█' * int(recall * 20)
    print(f'  {scenario:<30} {recall:.1%}  {bar}')

In [ ]:
# Top-1000 by anomaly score — what fraction are fraud?
feat_df['_score'] = score_samples(model, feat_df)
top1000 = feat_df.nlargest(1000, '_score')

print('=== Top-1000 highest-scored rows ===')
print(f'is_fraud rate:   {top1000["is_fraud"].fillna(False).mean():.1%}')
print(f'(Random baseline: ~{feat_df["is_fraud"].fillna(False).mean():.3%})')
print()
print('Fraud type breakdown in top-1000:')
print(top1000['fraud_type'].value_counts(dropna=False).head(10).to_string())

In [ ]:
# Score distribution: clean vs fraud
fig, ax = plt.subplots(figsize=(10, 4))

clean_scores = feat_df.loc[~feat_df['is_fraud'].fillna(False), '_score']
fraud_scores = feat_df.loc[feat_df['is_fraud'].fillna(False), '_score']

ax.hist(clean_scores.sample(min(50_000, len(clean_scores)), random_state=42),
        bins=60, alpha=0.6, label=f'Clean (n={len(clean_scores):,})', density=True, color='#2563EB')
ax.hist(fraud_scores,
        bins=60, alpha=0.8, label=f'Fraud (n={len(fraud_scores):,})', density=True, color='#DC2626')

ax.set_xlabel('Anomaly score (higher = more anomalous)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Isolation Forest: Score Distribution (Clean vs Fraud)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('../data/fraud_score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → data/fraud_score_distribution.png')

## Step 7 — Save Model

Saved via `joblib` (safe, fast, sklearn-native). The model is loaded at ML service
startup and used in `FraudPredictor.compute_user_fraud_penalty()`.

In [ ]:
from training.fraud_model import save, load

save(model)

# Smoke test: reload and score one row
model_loaded = load()
test_row     = feat_df[FEATURE_NAMES].iloc[0:1]
test_score   = score_samples(model_loaded, feat_df.iloc[0:1])
print(f'Reload OK. Test score: {float(test_score[0]):.4f}')

print()
print('Training pipeline complete.')
print(f'  Features parquet: {(DATA_DIR / "features_train.parquet").stat().st_size / 1e6:.0f} MB')
print(f'  Fraud model:      {(Path("../models/fraud_model.pkl")).stat().st_size / 1e3:.0f} KB')